[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/31_transformers/90_exercises.ipynb)

# Bài tập — Chương 31: Transformers (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/31_transformers

In [ ]:
import math
import time

import numpy as np
import pandas as pd

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

try:
    import torch
    from torch import nn
    from torch.nn.utils.rnn import pad_sequence
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
rng = np.random.default_rng(0)


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Công thức scaled dot-product attention; vì sao chia $\sqrt{d_k}$? (b) Vì sao Transformer cần mã hoá vị trí? (c) Encoder-only, decoder-only, encoder–decoder khác nhau ở mask và mục tiêu huấn luyện thế nào? (d) KV cache lưu gì, vì sao dùng được? (e) Trong MLM của BERT, token được chọn bị thay thế thế nào?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** (1) Nhóm A huấn luyện mô hình sinh văn bản: một encoder Transformer **hai chiều** dự đoán token tiếp theo tại mọi vị trí. Loss train giảm về gần 0 sau vài trăm bước, nhưng văn bản sinh ra vô nghĩa. (2) Nhóm B train một ViT **từ đầu** trên 2 000 ảnh X-quang và thấy kém hơn hẳn một CNN nhỏ; họ kết luận "Transformer không hợp với ảnh y tế". Giải thích và đề xuất.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Một query $\mathbf q = [1, 0]$, ba key $\mathbf k_1 = [1, 0]$, $\mathbf k_2 = [0, 1]$, $\mathbf k_3 = [1, 1]$, value vô hướng $v = [1, 2, 3]$, $d_k = 2$. Tính trọng số attention `w` (có chia $\sqrt{d_k}$), đầu ra `out`, và trọng số `w_noscale` khi **không** chia.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
scores = None
w = None
out = None
w_noscale = None
### END CODE HERE ###
print(f"w = {np.round(w, 4)} | out = {out:.4f} | w (không chia) = {np.round(w_noscale, 4)}")

In [ ]:
q3, K3, v3 = np.array([[1.0, 0.0]]), np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]), np.array([1.0, 2.0, 3.0])
check_close(w, softmax(q3 @ K3.T / np.sqrt(2))[0], message="trọng số có chia √d")
check_close(out, 2.0, message="đầu ra = 2 (k1, k3 cùng trọng số, đối xứng quanh v = 2)")
check_close(w_noscale, softmax(q3 @ K3.T)[0], message="trọng số không chia")
check_greater(w_noscale[0], w[0], "không chia √d → phân phối sắc hơn")

**Câu 4.** Đếm tham số (có bias, LayerNorm có weight + bias): (a) multi-head attention $d_{model} = 64$ → `n_mha`; (b) một khối Transformer encoder $d = 64$, FFN $d_{ff} = 256$, 2 LayerNorm → `n_block`; (c) GPT: từ vựng 100, $d = 64$, 2 khối như (b), ngữ cảnh 32 (position embedding học được), LayerNorm cuối, đầu ra **dùng chung** trọng số với embedding token → `n_gpt`.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
n_mha = None
n_block = None
n_gpt = None
### END CODE HERE ###
print(n_mha, n_block, n_gpt)

In [ ]:
count = lambda m: sum(p.numel() for p in m.parameters())
check_equal(n_mha, count(nn.MultiheadAttention(64, 4)), "MHA")
layer4 = nn.TransformerEncoderLayer(64, 4, 256)
check_equal(n_block, count(layer4), "khối encoder")


class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok, self.pos = nn.Embedding(100, 64), nn.Embedding(32, 64)
        self.blocks = nn.ModuleList([nn.TransformerEncoderLayer(64, 4, 256) for _ in range(2)])
        self.ln = nn.LayerNorm(64)
        self.head = nn.Linear(64, 100, bias=False)
        self.head.weight = self.tok.weight


check_equal(n_gpt, count(TinyGPT()), "GPT (parameters() không đếm trùng trọng số dùng chung)")

## L3 — Code

**Bài 5.** Viết `make_attention_mask(lens, T)` trả về tensor bool `(B, T, T)`, `True` = **được nhìn**: vị trí $i$ nhìn được vị trí $j$ khi $j \le i$ (causal) **và** $j$ < độ dài thật của chuỗi (không phải đệm). Dùng với `torch.nn.functional.scaled_dot_product_attention` (quy ước giống: `True` = tham gia attention).

In [ ]:
def make_attention_mask(lens, T):
    ### START CODE HERE ### (≈ 3 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
lens5 = torch.tensor([4, 2])
m5 = make_attention_mask(lens5, 4)
check_equal(tuple(m5.shape), (2, 4, 4), "shape")
check_true(torch.equal(m5[0], torch.tril(torch.ones(4, 4, dtype=torch.bool))), "chuỗi đủ dài: mask tam giác dưới")
check_true(torch.equal(m5[1], torch.tensor([[1, 0, 0, 0], [1, 1, 0, 0], [1, 1, 0, 0], [1, 1, 0, 0]], dtype=torch.bool)),
           "chuỗi dài 2: không nhìn vị trí đệm 2, 3")
torch.manual_seed(0)
q5, k5, v5 = (torch.randn(2, 4, 8) for _ in range(3))
out5 = nn.functional.scaled_dot_product_attention(q5, k5, v5, attn_mask=m5)
ref5 = nn.functional.scaled_dot_product_attention(q5[1:, :2], k5[1:, :2], v5[1:, :2], is_causal=True)
check_close(out5[1, :2], ref5[0], tol=1e-6, message="chuỗi 2 khớp attention trên phần thật (bỏ hẳn đệm)")

**Bài 6.** Viết `mha_forward(x, W_qkv, b_qkv, W_o, b_o, n_heads, causal=False)` (PyTorch, không dùng `nn.MultiheadAttention`): `x` (B, T, D); `W_qkv` (3D, D) chứa W_Q, W_K, W_V xếp chồng (quy ước `in_proj_weight` của PyTorch); chia đầu, scaled dot-product attention (có causal mask nếu `causal`), ghép đầu, chiếu ra. So khớp `nn.MultiheadAttention`.

In [ ]:
def mha_forward(x, W_qkv, b_qkv, W_o, b_o, n_heads, causal=False):
    B, T, D = x.shape
    d_k = D // n_heads
    ### START CODE HERE ### (≈ 9 dòng)
    out = None
    ### END CODE HERE ###
    return out

In [ ]:
torch.manual_seed(0)
ref6 = nn.MultiheadAttention(24, 3, batch_first=True)
x6 = torch.randn(2, 7, 24)
params6 = (ref6.in_proj_weight, ref6.in_proj_bias, ref6.out_proj.weight, ref6.out_proj.bias)
with torch.no_grad():
    check_close(mha_forward(x6, *params6, n_heads=3), ref6(x6, x6, x6)[0], tol=1e-5, message="khớp nn.MultiheadAttention")
    future6 = torch.triu(torch.ones(7, 7, dtype=torch.bool), diagonal=1)   # nn.MultiheadAttention: True = BỊ CHE
    check_close(mha_forward(x6, *params6, n_heads=3, causal=True), ref6(x6, x6, x6, attn_mask=future6)[0], tol=1e-5,
                message="khớp với causal mask")

## L4 — Debug

**Bài 7.** Một mini GPT ký tự (cho sẵn, giống bài 03) dùng hàm `causal_mask_buggy` — huấn luyện 300 bước cho loss train **thấp đáng ngờ**. Viết `causal_mask(T)` đúng (bool, `True` = được nhìn). So sánh hai phiên bản: loss train, "khoảng hở nhân quả" (đổi token từ vị trí 40 trở đi, logits trước vị trí 40 thay đổi bao nhiêu), và văn bản sinh ra.

In [ ]:
def causal_mask_buggy(T):
    return torch.triu(torch.ones(T, T, dtype=torch.bool))


def causal_mask(T):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
reviews7 = pd.read_csv(repo_path("data", "sample", "reviews_vi.csv"))["text"].tolist()
corpus7 = "\n".join(reviews7) + "\n"
chars7 = sorted(set(corpus7))
stoi7 = {c: i for i, c in enumerate(chars7)}
data7 = torch.tensor([stoi7[c] for c in corpus7])
V7, CTX7 = len(chars7), 64


class MaskedAttention(nn.Module):
    def __init__(self, d, h, mask):
        super().__init__()
        self.h, self.qkv, self.proj = h, nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = (t.view(B, T, self.h, D // self.h).transpose(1, 2) for t in self.qkv(x).split(D, -1))
        s = (q @ k.transpose(-2, -1) / math.sqrt(D // self.h)).masked_fill(~self.mask[:T, :T], float("-inf"))
        return self.proj((torch.softmax(s, -1) @ v).transpose(1, 2).reshape(B, T, D))


class MiniGPT(nn.Module):
    def __init__(self, mask_fn, d=64, h=4, n_layers=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V7, d), nn.Embedding(CTX7, d)
        self.attn = nn.ModuleList([MaskedAttention(d, h, mask_fn(CTX7)) for _ in range(n_layers)])
        self.mlp = nn.ModuleList([nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)) for _ in range(n_layers)])
        self.lns = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V7)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for attn, mlp, ln in zip(self.attn, self.mlp, self.lns):
            x = x + attn(ln(x))
            x = x + mlp(x)
        return self.head(self.ln(x))


def batch7(g, n=64):
    s = torch.randint(0, len(data7) - CTX7 - 1, (n,), generator=g)
    return (torch.stack([data7[i:i + CTX7] for i in s]).to(device), torch.stack([data7[i + 1:i + CTX7 + 1] for i in s]).to(device))


def train7(mask_fn, steps=300):
    torch.manual_seed(0)
    model = MiniGPT(mask_fn).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
    g = torch.Generator().manual_seed(0)
    for _ in range(steps):
        x, y = batch7(g)
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(x).reshape(-1, V7), y.reshape(-1))
        loss.backward()
        opt.step()
    return model.eval(), loss.item()


def causality_gap7(model):
    x, _ = batch7(torch.Generator().manual_seed(5), n=4)
    x2 = x.clone()
    x2[:, 40:] = (x2[:, 40:] + 7) % V7
    with torch.no_grad():
        return (model(x)[:, :40] - model(x2)[:, :40]).abs().max().item()


@torch.no_grad()
def sample7(model, n=60, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    idx = torch.tensor([[stoi7["\n"]]], device=device)
    for _ in range(n):
        probs = torch.softmax(model(idx[:, -CTX7:])[:, -1] / 0.7, -1)
        idx = torch.cat([idx, torch.multinomial(probs, 1, generator=g)], 1)
    return "".join(chars7[i] for i in idx[0].tolist())


results7 = {}
for name, fn in [("mask lỗi", causal_mask_buggy), ("mask đúng", causal_mask)]:
    model7, loss7 = train7(fn)
    results7[name] = (loss7, causality_gap7(model7))
    print(f"{name:9s}: loss train = {loss7:.3f} | khoảng hở nhân quả = {results7[name][1]:.2e} | sinh: {sample7(model7)!r}")

check_less(results7["mask lỗi"][0], 0.5 * results7["mask đúng"][0], "mask lỗi: loss train thấp bất thường (nhìn trộm)")
check_greater(results7["mask lỗi"][1], 1e-3, "mask lỗi: logits quá khứ phụ thuộc token tương lai")
check_less(results7["mask đúng"][1], 1e-5, "mask đúng: nhân quả")

## L5 — Phân tích

**Bài 8.** **Induction head.** Chuỗi 48 token ngẫu nhiên (64 ký hiệu); một đoạn A (độ dài **ngẫu nhiên** 6–19, bắt đầu ở vị trí 0–8) được **lặp lại ngay sau nó** — khoảng cách lặp thay đổi theo từng chuỗi. Trên phần lặp, token tiếp theo dự đoán được nếu mô hình "tìm lần xuất hiện trước của token hiện tại và lấy token đứng sau nó". Train GPT 1 khối và 2 khối (dùng `GPT8` cho sẵn), lưu `acc8[n_layers]` = accuracy trên các vị trí dự đoán được của phần lặp (1 000 chuỗi validation). Nhận xét.

In [ ]:
VOC8, T8 = 64, 48


def induction_batch(n, g):
    x = torch.randint(0, VOC8, (n, T8), generator=g)
    start = torch.randint(0, 9, (n,), generator=g)                    # vị trí bắt đầu đoạn A
    seg = torch.randint(6, 20, (n,), generator=g)                     # độ dài A ngẫu nhiên → khoảng cách lặp ngẫu nhiên
    target_mask = torch.zeros(n, T8 - 1, dtype=torch.bool)
    for i in range(n):
        p, la = start[i].item(), seg[i].item()
        x[i, p + la:p + 2 * la] = x[i, p:p + la]                       # lặp A ngay sau chính nó
        target_mask[i, p + la:p + 2 * la - 1] = True                   # vị trí có đích dự đoán được
    return x[:, :-1].to(device), x[:, 1:].to(device), target_mask.to(device)


class GPT8(nn.Module):
    def __init__(self, n_layers, d=64, h=4):
        super().__init__()
        self.tok, self.pos = nn.Embedding(VOC8, d), nn.Embedding(T8, d)
        layer = nn.TransformerEncoderLayer(d, h, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, VOC8)
        self.register_buffer("future", torch.triu(torch.ones(T8, T8, dtype=torch.bool), 1))

    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        return self.head(self.ln(self.blocks(h, mask=self.future[:T, :T], is_causal=True)))

In [ ]:
STEPS8 = 2500
acc8 = {}
### START CODE HERE ### (≈ 18 dòng)
# với n_layers ∈ {1, 2}: torch.manual_seed(0); model = GPT8(n_layers); AdamW lr 3e-3; STEPS8 bước, batch 64,
# cross-entropy trên mọi vị trí, clip 1.0; rồi acc8[n_layers] = accuracy trên vị trí target_mask của batch validation
pass
### END CODE HERE ###

In [ ]:
check_greater(acc8[2], 0.75, "2 khối học được cơ chế induction")
check_less(acc8[1], 0.4, "1 khối không làm được")
check_greater(acc8[2] - acc8[1], 0.4, "chênh lệch rõ")

## L6 — Thực tế

**Bài 9.** **Phiên sự kiện đáng ngờ, lần này với Transformer.** Dữ liệu như bài tập 9 chương 30: phiên 10–30 sự kiện (mã 0–5), luôn có một sự kiện 6 (đăng nhập thiết bị lạ) và một sự kiện 7 (đổi mật khẩu); nhãn 1 nếu 7 xảy ra **sau** 6. Viết `SessionTransformer(use_pos)`: embedding (PAD = 8, `padding_idx`), position embedding học được (nếu `use_pos`), `nn.TransformerEncoder` 2 khối ($d = 32$, 4 đầu, `src_key_padding_mask`), **trung bình có mask** trên các vị trí thật, lớp tuyến tính → 2 lớp. Train 20 epoch (AdamW lr 1e-3, batch 64), lưu `acc9[use_pos]`.

In [ ]:
def make_sessions(n, seed):
    r = np.random.default_rng(seed)
    seqs, labels = [], []
    for _ in range(n):
        L = r.integers(10, 31)
        s = [int(v) for v in r.integers(0, 6, L)]
        p1, p2 = sorted(r.choice(L + 2, 2, replace=False))
        suspicious = r.random() < 0.5
        a, b = (6, 7) if suspicious else (7, 6)
        s.insert(p1, a)
        s.insert(p2, b)
        seqs.append(s)
        labels.append(int(suspicious))
    return seqs, np.array(labels)


tr9, y_tr9 = make_sessions(2000, seed=10)
te9, y_te9 = make_sessions(1000, seed=11)
X_tr9 = pad_sequence([torch.tensor(s) for s in tr9], batch_first=True, padding_value=8)
X_te9 = pad_sequence([torch.tensor(s) for s in te9], batch_first=True, padding_value=8)

In [ ]:
class SessionTransformer(nn.Module):
    def __init__(self, use_pos, d=32):
        super().__init__()
        self.use_pos = use_pos
        ### START CODE HERE ### (≈ 5 dòng)
        pass
        ### END CODE HERE ###

    def forward(self, x):
        ### START CODE HERE ### (≈ 5 dòng)
        return None
        ### END CODE HERE ###


acc9 = {}
y9 = torch.tensor(y_tr9)
for use_pos in [False, True]:
    torch.manual_seed(0)
    model9 = SessionTransformer(use_pos).to(device)
    opt = torch.optim.AdamW(model9.parameters(), lr=1e-3)
    for _ in range(20):
        model9.train()
        perm = torch.randperm(len(X_tr9))
        for s in range(0, len(perm), 64):
            idx = perm[s:s + 64]
            opt.zero_grad()
            nn.functional.cross_entropy(model9(X_tr9[idx].to(device)), y9[idx].to(device)).backward()
            opt.step()
    model9.eval()
    with torch.no_grad():
        acc9[use_pos] = (model9(X_te9.to(device)).argmax(1).cpu().numpy() == y_te9).mean()
    print(f"{'có' if use_pos else 'không':5s} mã hoá vị trí: accuracy test = {acc9[use_pos]:.3f}")

In [ ]:
check_greater(acc9[True], 0.95, "Transformer có vị trí ≥ 0.95")
check_less(acc9[False], 0.6, "không có vị trí ≈ đoán ngẫu nhiên")
with torch.no_grad():
    a, b = model9(X_te9[:1].to(device)), model9(torch.cat([X_te9[:1], torch.full((1, 5), 8)], 1).to(device))
check_close(a.cpu(), b.cpu(), tol=1e-4, message="thêm đệm không đổi kết quả (mask + trung bình có mask đúng)")

## L7 — Nâng cao

**Bài 10.** **Attention theo khối với online softmax** (ý tưởng cốt lõi của FlashAttention). Viết `chunked_attention(Q, K, V, chunk)` (NumPy) xử lý key/value theo từng khối `chunk` cột, **không bao giờ** tạo ma trận $n_q\times n_k$ đầy đủ. Duy trì cho mỗi query: max hiện tại `m`, tổng chuẩn hoá `l`, bộ tích luỹ `acc`; với khối mới có điểm `S`: `m_new = max(m, max(S))`, hiệu chỉnh `l` và `acc` bởi `exp(m − m_new)`, cộng phần của khối mới. Kết quả phải khớp attention đầy đủ.

In [ ]:
def full_attention(Q, K, V):
    return softmax(Q @ K.T / np.sqrt(Q.shape[1])) @ V


def chunked_attention(Q, K, V, chunk):
    n_q, d = Q.shape
    m = np.full(n_q, -np.inf)
    l = np.zeros(n_q)
    acc = np.zeros((n_q, V.shape[1]))
    largest = 0                                                         # kích thước ma trận điểm lớn nhất từng tạo
    ### START CODE HERE ### (≈ 9 dòng)
    # for start in range(0, len(K), chunk): S = Q @ K[start:start+chunk].T / sqrt(d); cập nhật m, l, acc
    pass
    ### END CODE HERE ###
    return acc / l[:, None], largest

In [ ]:
Q10, K10, V10 = rng.normal(size=(300, 32)), rng.normal(size=(1000, 32)), rng.normal(size=(1000, 16))
K10[17] *= 50                                                           # một key có điểm rất lớn: kiểm tra ổn định số
out10, largest10 = chunked_attention(Q10, K10, V10, chunk=64)
check_close(out10, full_attention(Q10, K10, V10), tol=1e-10, message="khớp attention đầy đủ")
check_true(np.isfinite(out10).all(), "ổn định số")
check_equal(largest10, 300 * 64, "chỉ tạo ma trận điểm n_q × chunk")
for c in [1, 7, 1000]:
    check_close(chunked_attention(Q10, K10, V10, chunk=c)[0], out10, tol=1e-10, message=f"chunk = {c}")
print(f"ma trận điểm lớn nhất: {largest10:,} phần tử so với {300 * 1000:,} khi tính đầy đủ")